### DESC ELAsTiCC2 — Demo 2 : Boucle sur N light curves

Ce notebook s'inspire de `elasticc2_demo1.ipynb` et affiche **N light curves** sélectionnées
aléatoirement dans un intervalle de redshift donné, en les organisant dans une grille de subplots.

**Paramètres configurables (cellule `Parameters`) :**
- `N_CURVES` : nombre de light curves à afficher
- `Z_MIN`, `Z_MAX` : intervalle de redshift pour la sélection
- `OBJ_CLASS` : classe d'objet SNANA (ex. `'SNIa-SALT3'`)
- `FILE_NUM` : numéro du fichier PHOT à charger (1–40, `None` = tous)
- `MIN_DETECTIONS` : nombre minimum de détections exigé par objet
- `DETECTED_ONLY` : si `True`, n'affiche que les points détectés (`PHOTFLAG & photflag_detect != 0`)
- `NCOLS` : nombre de colonnes dans la grille de subplots
- `RANDOM_SEED` : graine pour la reproductibilité (`None` = aléatoire)


In [ ]:
%matplotlib inline

import sys
import os
import math
import pathlib
import logging

import numpy
import matplotlib
from matplotlib import pyplot

libdir = pathlib.Path(os.getcwd()).parent.parent / "lib_elasticc2"
sys.path.insert(0, str(libdir))
from transcientslightcurves import elasticc2_snana_reader

_logger = logging.getLogger("main")
if not _logger.hasHandlers():
    _logout = logging.StreamHandler(sys.stderr)
    _logger.addHandler(_logout)
    _logout.setFormatter(
        logging.Formatter(
            '[%(asctime)s - %(levelname)s] - %(message)s',
            datefmt='%Y-%m-%d %H:%M:%S'
        )
    )
_logger.setLevel(logging.INFO)
_logger.info("Initialisation terminée.")

## Paramètres

In [ ]:
# ── Paramètres principaux ────────────────────────────────────────────────────

N_CURVES        = 12          # Nombre de light curves à afficher
OBJ_CLASS       = 'SNIa-SALT3'  # Classe SNANA à utiliser
Z_MIN           = 0.1         # Borne inférieure du redshift
Z_MAX           = 0.5         # Borne supérieure du redshift
FILE_NUM        = 1           # Fichier PHOT à charger (1–40 ; None = tous)
MIN_DETECTIONS  = 5           # Nombre minimum de détections par objet
DETECTED_ONLY   = True        # True : points détectés uniquement
NCOLS           = 3           # Nombre de colonnes dans la grille
RANDOM_SEED     = 42          # None pour aléatoire

# Chemin vers les données ELAsTiCC2
DATA_DIR    = "/Users/dagoret/DATA/DESC_TD_PUBLIC/ELASTICC/ELASTICC2_TRAINING_SAMPLE_2"
DIR_PREFIX  = "ELASTICC2_TRAIN_02_"

# ────────────────────────────────────────────────────────────────────────────
NROWS = math.ceil(N_CURVES / NCOLS)
rng = numpy.random.default_rng(seed=RANDOM_SEED)
print(f"Grille : {NROWS} lignes × {NCOLS} colonnes pour {N_CURVES} light curves")

## Chargement du lecteur SNANA

In [ ]:
esr = elasticc2_snana_reader(DATA_DIR, dir_prefix=DIR_PREFIX)
print("Classes disponibles :")
print("\n".join(esr.obj_class_names))

## Chargement des fichiers HEAD et Truth

In [ ]:
_logger.info(f"Lecture des fichiers HEAD pour {OBJ_CLASS}...")
head = esr.get_head(OBJ_CLASS, return_format='pandas')
_logger.info("...terminé.")
print(f"{len(head)} entrées HEAD lues pour {OBJ_CLASS}")

In [ ]:
_logger.info(f"Lecture du fichier truth pour {OBJ_CLASS}...")
truth = esr.get_object_truth(OBJ_CLASS, return_format='pandas')
_logger.info("...terminé.")
print(f"{len(truth)} entrées truth lues pour {OBJ_CLASS}")

## Chargement des light curves (fichier PHOT)

In [ ]:
_logger.info(f"Chargement des light curves (file_num={FILE_NUM})...")
all_ltcvs = esr.get_all_ltcvs(OBJ_CLASS, file_num=FILE_NUM, return_format='pandas')
_logger.info("...terminé.")
print(f"{all_ltcvs['SNID'].nunique()} objets chargés en mémoire")

## Sélection aléatoire de N objets

In [ ]:
# Compter les détections par objet
detcounts = (
    all_ltcvs[(all_ltcvs['PHOTFLAG'] & esr.photflag_detect) != 0]
    .groupby('SNID').agg('count')['MJD']
    .reset_index()
    .rename({'MJD': 'ndetect'}, axis=1)
)

# Joindre avec la table truth
truth_with_counts = truth.join(detcounts.set_index('SNID'), on='SNID', how='inner')

# Filtrer sur le redshift et le nombre de détections
subset = truth_with_counts[
    (truth_with_counts['ZCMB'] >= Z_MIN) &
    (truth_with_counts['ZCMB'] <  Z_MAX) &
    (truth_with_counts['ndetect'] >= MIN_DETECTIONS)
].copy()

print(f"{len(subset)} objets satisfont les critères (z∈[{Z_MIN},{Z_MAX}), ndetect≥{MIN_DETECTIONS})")

if len(subset) < N_CURVES:
    print(f"⚠️  Seulement {len(subset)} objets disponibles — N_CURVES réduit à {len(subset)}")
    N_CURVES = len(subset)
    NROWS = math.ceil(N_CURVES / NCOLS)

# Tirage aléatoire sans remise
chosen_indices = rng.choice(len(subset), size=N_CURVES, replace=False)
chosen_snids   = subset['SNID'].values[chosen_indices]
print(f"SNIDs sélectionnés : {chosen_snids}")

## Affichage en grille des N light curves

In [ ]:
# Couleurs par bande LSST
PLOTCOLORS = {
    'u': '#cc0ccc',
    'g': '#00cc44',
    'r': '#cc0000',
    'i': '#ff4400',
    'z': '#886600',
    'Y': '#442200'
}

FIG_WIDTH  = 5.5   # largeur d'un subplot en pouces
FIG_HEIGHT = 4.0   # hauteur d'un subplot en pouces

fig, axes = pyplot.subplots(
    NROWS, NCOLS,
    figsize=(FIG_WIDTH * NCOLS, FIG_HEIGHT * NROWS),
    tight_layout=True
)
# Garantir un tableau 1D même si NROWS==1 ou NCOLS==1
axes_flat = numpy.array(axes).flatten()

for idx, snid in enumerate(chosen_snids):
    ax = axes_flat[idx]

    # Extraire la light curve depuis le DataFrame global
    ltcv = all_ltcvs[all_ltcvs['SNID'] == snid]

    if DETECTED_ONLY:
        ltcv = ltcv[(ltcv['PHOTFLAG'] & esr.photflag_detect) != 0]

    # Infos de la vérité terrain
    obj_truth = truth[truth['SNID'] == snid]
    z_val     = obj_truth['ZCMB'].values[0] if len(obj_truth) > 0 else float('nan')
    ndet      = subset[subset['SNID'] == snid]['ndetect'].values[0]

    # Bandes présentes
    knownbands  = ltcv['BAND'].unique()
    bandstoplot = [b for b in PLOTCOLORS if b in knownbands]

    for band in bandstoplot:
        bltcv = ltcv[ltcv['BAND'] == band]
        ax.errorbar(
            bltcv['MJD'], bltcv['FLUXCAL'], yerr=bltcv['FLUXCALERR'],
            color=PLOTCOLORS[band], linestyle='None', marker='o',
            markersize=3, capsize=2, label=band
        )

    ymin, ymax = ax.get_ylim()
    if ymin > 0:
        ax.set_ylim(0, ymax)

    ax.set_title(f"SNID {snid}\nz={z_val:.3f}  ndet={ndet}", fontsize=9)
    ax.set_xlabel("MJD", fontsize=8)
    ax.set_ylabel("FLUXCAL", fontsize=8)
    ax.tick_params(axis='both', labelsize=7)
    ax.legend(fontsize=7, ncol=3, loc='upper right')

# Masquer les axes inutilisés si N_CURVES < NROWS*NCOLS
for idx in range(N_CURVES, len(axes_flat)):
    axes_flat[idx].set_visible(False)

detected_label = "détectés uniquement" if DETECTED_ONLY else "tous les points"
fig.suptitle(
    f"{OBJ_CLASS} — {N_CURVES} light curves aléatoires\n"
    f"z ∈ [{Z_MIN}, {Z_MAX})  |  ndet ≥ {MIN_DETECTIONS}  |  {detected_label}  |  seed={RANDOM_SEED}",
    fontsize=11, y=1.01
)
pyplot.tight_layout()
pyplot.show();

## (Optionnel) Distribution en redshift des objets sélectionnés

In [ ]:
chosen_z = truth[truth['SNID'].isin(chosen_snids)]['ZCMB'].values

fig2, ax2 = pyplot.subplots(1, 1, figsize=(6, 3), tight_layout=True)
ax2.hist(chosen_z, bins=numpy.linspace(Z_MIN, Z_MAX, 15), color='steelblue', edgecolor='white')
ax2.set_xlabel(r"$z_{\rm CMB}$")
ax2.set_ylabel("N")
ax2.set_title(f"Distribution en redshift des {N_CURVES} objets sélectionnés ({OBJ_CLASS})")
pyplot.show();